# **Data Preparation**

## **Select Data**

### Data Selection During Extraction

During the data extraction stage, we significantly reduced the original GitHub Archive schema to only the fields relevant to our analysis goals.

From the original event schema, we selected the following attributes:
- repo_name (repo.name)
- user_login (actor.login)
- event_type (type)
- event_time (created_at → timestamp)
- repo_url (repo.url)
- public
- season (derived)

The following fields were excluded:

- id → not needed for aggregation or modeling
- actor.id, actor.display_login, actor.gravatar_id, actor.url, actor.avatar_url → redundant user metadata
- repo.id → not required at this stage (can be reintroduced later if needed)
- payload → excluded due to very large size and high processing cost
- org.* → not relevant for current analysis
- other nested or rarely used fields → not relevant for the task

Additionally:
- Only selected event types were included:
  - ForkEvent
  - PullRequestEvent
  - WatchEvent
  - PushEvent
  - IssuesEvent
  - IssueCommentEvent
  - ReleaseEvent

- Time filtering was applied:
  - Two seasonal windows (H1 and H2 (first half vs second half of the year)) to reduce seasonality bias

- Repository filtering:
  - Only repositories with forks > 5 in the anchor window were retained
  - Only repositories with at least one non-fork activity signal (watches, pushes, issues, issue comments, or releases) were retained

These decisions were made to reduce data volume, control costs, and focus on features relevant to repository activity and contribution behavior.

### Data Selection During EDA

During exploratory data analysis, additional observations led to further selection decisions:

- The `public` field showed no variation (almost all repositories are public), therefore it is not informative for modeling and can be excluded.

- Extremely large repositories (outliers) were identified in:
  - watches
  - pushes
  - issues
  - forks / PRs

  These repositories dominate distributions and distort analysis, therefore they will be handled during the Data Cleaning stage.

- Seasonal split (H1 vs H2) was validated and preserved: this will later be used to ensure balanced sampling.

At this point we still keep every sampled repository–season row; narrowing happens through feature choice and joins, not by dropping keys from the sample.

In [1]:
import pandas as pd

activity = pd.read_parquet('data/activity_sample.parquet')
fork_pr = pd.read_parquet('data/fork_pr_sample.parquet')

In [2]:
# Attribute selection for modeling
_drop = [c for c in ["public", "repo_url"] if c in activity.columns]
prep_activity = activity.drop(columns=_drop, errors="ignore").copy()
print("Dropped for modeling:", _drop)
print("prep_activity columns:", prep_activity.columns.tolist())


Dropped for modeling: ['public', 'repo_url']
prep_activity columns: ['repo_name', 'season', 'watches', 'pushes', 'issues', 'issue_comments', 'releases']


## **Clean Data**

### Data Cleaning Plan

Based on EDA findings, we address data quality as follows:

- Outliers on activity metrics (`watches`, `pushes`, `issues`, `issue_comments`, `releases`) and on fork/PR volumes: we prefer capping (winsorization) at a high quantile rather than deleting repositories, unless a future rule explicitly flags bad data. Threshold-based row removal remains an alternative if modeling experiments require it.

- Event timestamps: fork and PR times are used for matching in Construct Data; we require PR time after fork time and inside the observation window. Full calendar validation beyond the extract is left for extended QA if needed.

These steps aim to stabilize downstream modeling while respecting that GitHub activity is naturally heavy-tailed.

In code we winsorize the five activity counters at the 99th percentile (`WINSOR_Q`). Original columns stay in `prep_activity`; capped copies use the `*_winsor` suffix. Fork and PR list lengths are not winsorized here because labels are derived from the raw event lists before aggregation.

In [3]:
ACTIVITY_COUNT_COLS = ["watches", "pushes", "issues", "issue_comments", "releases"]
WINSOR_Q = 0.99


def winsor_upper(s: pd.Series, q: float = WINSOR_Q) -> pd.Series:
    # Float64: quantile() is float; Int64 columns cannot store non-integer clipped values
    x = s.astype("float64")
    return x.clip(lower=0, upper=x.quantile(q))


prep_activity = prep_activity.copy()
for c in ACTIVITY_COUNT_COLS:
    if c in prep_activity.columns:
        prep_activity[c + "_winsor"] = winsor_upper(prep_activity[c])

# Quick check — maxima after capping
prep_activity[[c + "_winsor" for c in ACTIVITY_COUNT_COLS if c in prep_activity.columns]].describe().T[["max"]]


,max
watches_winsor,1536.02
pushes_winsor,1192.02
issues_winsor,301.00
issue_comments_winsor,1169.01
releases_winsor,33.00


## **Construct Data**


### Construct Data Plan

Several transformations and feature engineering steps are required:

#### Sampling Strategy
To address sampling bias across repository sizes:

- Apply stratified sampling based on activity/popularity:
  - low activity
  - medium activity
  - high activity

- Ensure equal representation across seasons

The BigQuery step in Collect Initial Data already applies filters and a random 100k cap. The bullets above describe how train/validation (or an optional resample) can follow this plan in the modeling phase.

#### Matching Forks to Pull Requests

A fork is considered "converted" if:

- A Pull Request exists
- From the same user (user_login)
- Within a defined time window (e.g. 90 days after the fork)

We do not count a pull request when:

- It occurs on or before the fork time, or
- It occurs after the observation window ends (e.g. more than 90 days after the fork if OBSERVATION_DAYS = 90).

#### Derived Features

New features to be created:

- fork_conversion_rate: share of forks that have at least one qualifying PR in the window (not the same as “all PR events divided by fork count” unless you define that ratio separately).
- Activity score (e.g. combination of watches, pushes, issues) — optional; not in the baseline code below.
- Possible extensions:
  - PR per user
  - forks per user
- log1p_* on raw and winsorized activity columns (see code cells).

#### Generated Records

- Only valid fork to PR pairs inside the window increase the converted-fork count.
- The export is one row per repository and season (one observation per repo_name and season) with scalar counts and rates; nested forks / prs arrays are not kept in the flat file.
- Rows stay in the table even when the PR list is empty (conversion can be zero).


In [4]:
# Construct — labels from fork / PR event lists (same keys as `activity`)
import numpy as np

OBSERVATION_DAYS = 90
KEYS = ["repo_name", "season"]


def as_event_list(cell):
    # Normalize one parquet cell
    if cell is None or (isinstance(cell, float) and np.isnan(cell)):
        return []
    arr = np.asarray(cell, dtype=object)
    if arr.size == 0:
        return []
    out = []
    for x in arr.ravel():
        if x is None or x is pd.NA:
            continue
        if isinstance(x, dict):
            out.append(x)
            continue
        try:
            names = getattr(x.dtype, "names", None)
            if names:
                out.append({k: x[k] for k in names})
                continue
        except Exception:
            pass
        try:
            item = x.item()
            if isinstance(item, dict):
                out.append(item)
        except Exception:
            pass
    return out


def count_converted_forks(fork_events, pr_events, obs_days):
    # Per-fork conversion: same user_login, PR strictly after fork, within obs_days
    if not fork_events:
        return 0, 0
    pr_by_user = {}
    for pr in pr_events:
        login = pr.get("user_login")
        if login is None:
            continue
        pr_by_user.setdefault(login, []).append(pd.Timestamp(pr["event_time"]))
    for ts_list in pr_by_user.values():
        ts_list.sort()
    n_forks = len(fork_events)
    converted = 0
    window = pd.Timedelta(days=obs_days)
    for fk in fork_events:
        login = fk.get("user_login")
        if login is None:
            continue
        t0 = pd.Timestamp(fk["event_time"])
        upper = t0 + window
        for pr_t in pr_by_user.get(login, []):
            if t0 < pr_t <= upper:
                converted += 1
                break
    return n_forks, converted


def build_labels(fork_pr_df, obs_days):
    # One label row per repo_name / season
    rows = []
    for row in fork_pr_df.itertuples(index=False):
        forks = as_event_list(row.forks)
        prs = as_event_list(row.prs)
        nf, nc = count_converted_forks(forks, prs, obs_days)
        rate = nc / nf if nf else np.nan
        rows.append(
            {
                "repo_name": row.repo_name,
                "season": row.season,
                "n_forks": nf,
                "n_converted_forks": nc,
                "fork_conversion_rate": rate,
                "n_pr_events": len(prs),
            }
        )
    return pd.DataFrame(rows)


labels = build_labels(fork_pr, OBSERVATION_DAYS)
labels[["n_forks", "n_converted_forks", "fork_conversion_rate"]].describe()

,n_forks,n_converted_forks,fork_conversion_rate
count,100000.000000,100000.000000,100000.000000
mean,29.819910,3.020300,0.132753
std,156.692666,20.679535,0.221853
min,6.000000,0.000000,0.000000
25%,7.000000,0.000000,0.000000
50%,11.000000,0.000000,0.000000
75%,20.000000,2.000000,0.166667
max,14513.000000,3399.000000,1.000000


In [5]:
# Construct log1p features
for c in ACTIVITY_COUNT_COLS:
    if c in prep_activity.columns:
        prep_activity["log1p_" + c] = np.log1p(prep_activity[c].astype(float))
    wc = c + "_winsor"
    if wc in prep_activity.columns:
        prep_activity["log1p_" + wc] = np.log1p(prep_activity[wc].astype(float))

# Feature names created
sorted([x for x in prep_activity.columns if x.startswith("log1p_")])


['log1p_issue_comments',
 'log1p_issue_comments_winsor',
 'log1p_issues',
 'log1p_issues_winsor',
 'log1p_pushes',
 'log1p_pushes_winsor',
 'log1p_releases',
 'log1p_releases_winsor',
 'log1p_watches',
 'log1p_watches_winsor']

## **Integrate Data**

### Data Integration Plan

The analytical table combines:

- activity_sample metrics (after selection and cleaning in prep_activity), and
- fork_pr_sample - derived label columns (`n_forks`, `n_converted_forks`, `fork_conversion_rate`, `n_pr_events`).

Integration logic:

- join on repo_name and season;
- use an inner join with a one-to-one key check.

After integration we have scalar counts and rates instead of nested event arrays in the modeling frame. Each row is one repository–season observation (not a single row collapsed across both seasons for the same repository). We keep season in the export so we can hold out a half-year block or stratify; it can be dropped later if a model is trained on pooled seasons only.

The merged result is modeling_df, which then will be reordered and written to disk.


In [6]:
# Integrate inner join so every feature row has exactly one label row
modeling_df = prep_activity.merge(labels, on=KEYS, how="inner", validate="one_to_one")

# Same row count as inputs
assert modeling_df.shape[0] == prep_activity.shape[0] == labels.shape[0]
print("modeling_df:", modeling_df.shape)
modeling_df.head()


modeling_df: (100000, 26)


,repo_name,season,watches,pushes,issues,issue_comments,releases,watches_winsor,pushes_winsor,issues_winsor,...,log1p_issues,log1p_issues_winsor,log1p_issue_comments,log1p_issue_comments_winsor,log1p_releases,log1p_releases_winsor,n_forks,n_converted_forks,fork_conversion_rate,n_pr_events
0,GoodRequest/BackendAssignment-Fitness,winter_spring,0,1,0,0,0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,8,0,0.000000,1
1,NREL/DE,winter_spring,0,1,0,0,0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,7,0,0.000000,0
2,danhpaiva/class-psc-algorithm-00,summer_autumn,0,1,0,0,0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,26,0,0.000000,0
3,NaturalCoder/TI96,winter_spring,0,25,0,0,0,0.0,25.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,12,11,0.916667,36
4,josedom24/ic-html5,winter_spring,0,2,0,0,0,0.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,13,0,0.000000,0


## **Format Data**

### Data Formatting Plan

Final formatting steps:

- Add a numeric 'repo_id' (category codes for `repo_name`) for tools that prefer integer identifiers; `repo_name` remains for traceability.
- Enforce column order: identifiers and `season`, then `log1p_*` feature columns, then label columns (`n_forks`, `n_converted_forks`, `fork_conversion_rate`, `n_pr_events`, `has_conversion`), then any remaining diagnostic fields.
- Shuffle rows with a fixed random seed to avoid accidental ordering bias.
- Ensure dtypes are numeric where possible; raw timestamps inside nested events are not carried into the flat export.

These are syntactic conveniences; they do not change the meaning of the aggregated values.


In [57]:
# Format identifiers and binary outcome for classifiers
modeling_df["repo_id"] = pd.Categorical(modeling_df["repo_name"]).codes
modeling_df["has_conversion"] = (modeling_df["n_converted_forks"] > 0).astype(int)

# Column groups: meta, log1p features, labels, then anything else
META = ["repo_name", "repo_id", "season"]
FEAT = sorted([c for c in modeling_df.columns if c.startswith("log1p_")])
LBL = [
    "n_forks",
    "n_converted_forks",
    "fork_conversion_rate",
    "n_pr_events",
    "has_conversion",
]
REST = [c for c in modeling_df.columns if c not in META + FEAT + LBL]

# Shuffle fixed seed for reproducibility
modeling_final = (
    modeling_df[META + FEAT + LBL + REST]
    .sample(frac=1.0, random_state=335)
    .reset_index(drop=True)
)

# Export full dataset for the Modeling phase
import os

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
path_pq = os.path.join(DATA_DIR, "modeling_dataset.parquet")
path_csv = os.path.join(DATA_DIR, "modeling_dataset.csv")

try:
    modeling_final.to_parquet(path_pq, index=False)
    print("Saved", path_pq, modeling_final.shape)
except ImportError:
    modeling_final.to_csv(path_csv, index=False)
    print("Saved", path_csv, "(install pyarrow for parquet)", modeling_final.shape)


Saved modeling_dataset.parquet (100000, 28)
